In [1]:
import sys, os
sys.path.append(os.path.abspath("../data"))

import models.gcn_model as gm
print(dir(gm))

d:\GitHub\FedGNN-CKD\fedgnn_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


['F', 'GCN', 'GCNConv', '__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__spec__', 'torch']


In [2]:
import torch
import copy
import os
import sys
from sklearn.metrics import accuracy_score

sys.path.append(os.path.abspath("../data"))
from models.gcn_model import GCN

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cpu


In [3]:
def load_graph(path):
    return torch.load(path, weights_only=False).to(device)

In [4]:
def evaluate(model, graph):
    model.eval()
    with torch.no_grad():
        out = model(graph.x, graph.edge_index)
        preds = out.argmax(dim=1).cpu()
        labels = graph.y.cpu()
    return accuracy_score(labels, preds)

In [5]:
def apply_ldp(model, epsilon, alpha=0.1):
    noisy_model = copy.deepcopy(model)
    noisy_model.eval()

    for name, param in noisy_model.named_parameters():
        if not param.requires_grad:
            continue

        param_std = torch.std(param.data)

        # More noise to first GNN layer
        if "conv1" in name:
            noise_std = alpha * param_std / epsilon
        else:
            noise_std = (alpha / 2) * param_std / epsilon

        noise = torch.normal(
            mean=0.0,
            std=noise_std,
            size=param.data.size()
        ).to(device)

        param.data += noise

    return noisy_model

In [6]:
def select_best_epsilon(clean_model, graph, epsilons):
    clean_acc = evaluate(clean_model, graph)
    print(f"Clean Accuracy: {clean_acc:.4f}")

    best_eps = None
    best_gap = float("inf")
    best_model = None

    for eps in epsilons:
        noisy_model = apply_ldp(clean_model, eps)
        noisy_acc = evaluate(noisy_model, graph)

        acc_gap = abs(clean_acc - noisy_acc)

        print(
            f"ε={eps} | Noisy Acc={noisy_acc:.4f} | "
            f"Gap={acc_gap:.4f}"
        )

        # Select model closest to clean accuracy
        if acc_gap < best_gap:
            best_gap = acc_gap
            best_eps = eps
            best_model = noisy_model

    print(
        f"Selected ε={best_eps} "
        f"(Accuracy gap={best_gap:.4f})\n"
    )

    return best_model, best_eps

In [8]:
if __name__ == "__main__":

    epsilons = [1.5, 2, 3, 4, 5]

    # Load graphs
    graph_A = load_graph("../data/graph/graph_A.pt")
    graph_B = load_graph("../data/graph/graph_B.pt")
    graph_C = load_graph("../data/graph/graph_C.pt")

    input_dim = graph_A.num_node_features

    model_A = GCN(input_dim, 32, 2).to(device)
    model_B = GCN(input_dim, 32, 2).to(device)
    model_C = GCN(input_dim, 32, 2).to(device)


    model_A.load_state_dict(torch.load("../data/models/model_A.pth"))
    model_B.load_state_dict(torch.load("../data/models/model_B.pth"))
    model_C.load_state_dict(torch.load("../data/models/model_C.pth"))

    # Select best ε per hospital
    model_A_ldp, eps_A = select_best_epsilon(model_A, graph_A, epsilons)
    model_B_ldp, eps_B = select_best_epsilon(model_B, graph_B, epsilons)
    model_C_ldp, eps_C = select_best_epsilon(model_C, graph_C, epsilons)

    # Save ONLY selected LDP models
    os.makedirs("../data/models", exist_ok=True)

    torch.save(model_A_ldp.state_dict(), "../data/models/model_A_ldp.pth")
    torch.save(model_B_ldp.state_dict(), "../data/models/model_B_ldp.pth")
    torch.save(model_C_ldp.state_dict(), "../data/models/model_C_ldp.pth")

    print("Best LDP models sent to global server")

Clean Accuracy: 0.9875
ε=1.5 | Noisy Acc=0.9875 | Gap=0.0000
ε=2 | Noisy Acc=0.9875 | Gap=0.0000
ε=3 | Noisy Acc=0.9875 | Gap=0.0000
ε=4 | Noisy Acc=0.9875 | Gap=0.0000
ε=5 | Noisy Acc=0.9875 | Gap=0.0000
Selected ε=1.5 (Accuracy gap=0.0000)

Clean Accuracy: 0.9830
ε=1.5 | Noisy Acc=0.9830 | Gap=0.0000
ε=2 | Noisy Acc=0.9830 | Gap=0.0000
ε=3 | Noisy Acc=0.9830 | Gap=0.0000
ε=4 | Noisy Acc=0.9830 | Gap=0.0000
ε=5 | Noisy Acc=0.9830 | Gap=0.0000
Selected ε=1.5 (Accuracy gap=0.0000)

Clean Accuracy: 1.0000
ε=1.5 | Noisy Acc=1.0000 | Gap=0.0000
ε=2 | Noisy Acc=1.0000 | Gap=0.0000
ε=3 | Noisy Acc=1.0000 | Gap=0.0000
ε=4 | Noisy Acc=1.0000 | Gap=0.0000
ε=5 | Noisy Acc=1.0000 | Gap=0.0000
Selected ε=1.5 (Accuracy gap=0.0000)

Best LDP models sent to global server
